# 파이썬 머신러닝 완벽 가이드 5장

회귀: 회귀 소개, 단순 선형 회귀, 경사 하강법, LinearRegression을 이용한 보스턴 주택 가격 예측, 다항 회귀와 과적합/과소적합 (p.308~337)

## 실습 환경 설정

- 사이킷런 1.2부터 `LinearRegression`의 `normalize` 파라미터가 삭제되었다. 책 p.321의 클래스 선언에서 `normalize`를 빼고 `fit_intercept`, `n_jobs`만 지정한다.
- 사이킷런 1.2부터 `load_boston()`이 삭제되었다. 책 p.324의 데이터 로딩은 CMU 원본 데이터를 직접 읽어오는 코드로 바꿔 적는다.

In [ ]:
import sklearn
print(sklearn.__version__)

## 01. 회귀 소개

### 개념 정리

**회귀(Regression)** 는 여러 개의 독립변수와 한 개의 종속변수 간의 상관관계를 모델링하는 기법이다. 영국의 통계학자 갈톤이 부모와 자식의 키를 연구하며, 자식의 키가 결국 평균으로 돌아가려는(회귀하려는) 경향을 발견한 데서 이름이 유래했다.

$$Y = W_1 X_1 + W_2 X_2 + \cdots + W_n X_n$$

- $Y$: 종속변수(예: 아파트 가격), $X_1, \dots, X_n$: 독립변수(방 개수, 크기, 주변 학군 등)
- $W_1, \dots, W_n$: 독립변수의 값에 영향을 미치는 **회귀 계수(Regression coefficients)**
- 머신러닝 관점에서 독립변수는 **피처**, 종속변수는 **결정 값**이다.
- 회귀 예측의 핵심은 주어진 피처와 결정 값 데이터 기반에서 학습을 통해 **최적의 회귀 계수를 찾아내는 것**이다.

**회귀의 유형**

| 기준 | 유형 |
|---|---|
| 회귀 계수의 결합 형태 | 선형 회귀 / 비선형 회귀 |
| 독립변수의 개수 | 1개: 단일 회귀 / 여러 개: 다중 회귀 |

**분류 vs 회귀**
- 분류는 예측값이 카테고리와 같은 **이산형 클래스 값**이고, 회귀는 **연속형 숫자 값**이다.

**대표적인 선형 회귀 모델**
- 선형 회귀는 실제 값과 예측값의 차이(오류의 제곱 값)를 최소화하는 직선형 회귀선을 최적화하는 방식이다.
- **규제(Regularization)**: 일반적인 선형 회귀의 과적합 문제를 해결하기 위해 회귀 계수에 페널티 값을 적용하는 것이다.

| 모델 | 특징 |
|---|---|
| 일반 선형 회귀 | 예측값과 실제 값의 RSS(Residual Sum of Squares)를 최소화하도록 회귀 계수를 최적화. 규제를 적용하지 않음 |
| 릿지(Ridge) | 선형 회귀에 **L2 규제** 추가. 상대적으로 큰 회귀 계수 값의 예측 영향도를 감소시키기 위해 회귀 계수값을 더 작게 만듦 |
| 라쏘(Lasso) | 선형 회귀에 **L1 규제** 적용. 예측 영향력이 작은 피처의 회귀 계수를 0으로 만들어 회귀 예측 시 피처가 선택되지 않게 함 → **피처 선택** 기능 |
| 엘라스틱넷(ElasticNet) | L2, L1 규제를 함께 결합. 피처가 많은 데이터 세트에서 적용되며, L1 규제로 피처 개수를 줄이고 L2 규제로 계수 값의 크기를 조정 |
| 로지스틱 회귀(Logistic Regression) | 이름은 회귀이지만 사실은 **분류**에 사용되는 선형 모델. 이진 분류뿐 아니라 텍스트 분류 같은 영역에서도 뛰어난 성능 |

## 02. 단순 선형 회귀를 통한 회귀 이해

### 개념 정리

**단순 선형 회귀**는 독립변수도 하나, 종속변수도 하나인 선형 회귀이다. 예를 들어 주택 가격이 주택의 크기로만 결정된다고 하면, 예측 함수는 다음과 같은 1차 함수로 모델링할 수 있다.

$$\hat{Y} = w_0 + w_1 X$$

- $w_0$: 절편(intercept), $w_1$: 기울기(회귀 계수)
- 실제 값 $Y$는 이 1차 함수 값에서 실제 값만큼의 오류 값을 더한 값, 즉 $w_0 + w_1 X + \text{오류 값}$이 된다.
- 실제 값과 회귀 모델의 차이에 따른 오류 값을 **잔차**라고 부른다.
- 최적의 회귀 모델을 만든다는 것은 **전체 데이터의 잔차(오류 값) 합이 최소가 되는 모델**을 만든다는 의미이며, 동시에 오류 값 합이 최소가 될 수 있는 최적의 회귀 계수를 찾는다는 의미이다.

**오류 합 계산 방법**
- 오류 값은 +나 -가 될 수 있어 단순히 더하면 오류 합이 크게 줄어들 수 있다.
- 따라서 오류 값의 절댓값을 취해서 더하거나(**Mean Absolute Error**), 오류 값의 제곱을 구해서 더하는 방식(**RSS, Residual Sum of Squares**)을 취한다.
- 일반적으로 미분 등의 계산을 편리하게 하기 위해 **RSS 방식**으로 오류 합을 구한다.

$$RSS(w_0, w_1) = \frac{1}{N} \sum_{i=1}^{N} \left( y_i - (w_0 + w_1 x_i) \right)^2$$

- 회귀에서 RSS는 $w$ 변수(회귀 계수)가 중심 변수임을 인지하는 것이 중요하다. 학습 데이터로 입력되는 독립변수와 종속변수는 RSS에서 모두 **상수**로 간주한다.
- 회귀에서 이 RSS는 **비용(Cost)** 이며, $w$ 변수로 구성되는 RSS를 **비용 함수** 또는 **손실 함수(loss function)** 라고 한다.
- 머신러닝 회귀 알고리즘은 데이터를 계속 학습하면서 이 비용 함수가 반환하는 값(오류 값)을 지속해서 감소시키고, 최종적으로 더 이상 감소하지 않는 최소의 오류 값을 구하는 것이다.

## 03. 비용 최소화하기 - 경사 하강법(Gradient Descent) 소개

### 개념 정리

- $w$ 파라미터의 개수가 적다면 고차원 방정식으로 비용 함수가 최소가 되는 $w$ 변숫값을 도출할 수 있지만, 파라미터가 많으면 고차원 방정식으로 문제를 해결하기 어렵다.
- **경사 하강법**은 고차원 방정식에 대한 문제를 해결해 주면서 비용 함수 RSS를 최소화하는 방법을 **직관적으로** 제공한다.
- '데이터를 기반으로 알고리즘이 스스로 학습한다'는 머신러닝의 개념을 가능하게 만들어준 핵심 기법 중 하나이다.
- 점진적으로 반복적인 계산을 통해 $w$ 파라미터 값을 업데이트하면서 오류 값이 최소가 되는 $w$ 파라미터를 구한다.

**핵심 아이디어**
- 비용 함수가 포물선 형태의 2차 함수라면, 최초 $w$에서부터 미분을 적용한 뒤 이 미분 값이 계속 감소하는 방향으로 순차적으로 $w$를 업데이트한다.
- 더 이상 미분된 1차 함수의 기울기가 감소하지 않는 지점을 비용 함수가 최소인 지점으로 간주하고 그때의 $w$를 반환한다.

**편미분 결과**

$R(w)$는 $w_0, w_1$ 두 개의 파라미터를 가지므로 각각 편미분한다.

$$\frac{\partial R(w)}{\partial w_1} = \frac{2}{N} \sum_{i=1}^{N} -x_i \left( y_i - (w_0 + w_1 x_i) \right) = -\frac{2}{N} \sum_{i=1}^{N} x_i \cdot (\text{실제값}_i - \text{예측값}_i)$$

$$\frac{\partial R(w)}{\partial w_0} = \frac{2}{N} \sum_{i=1}^{N} -\left( y_i - (w_0 + w_1 x_i) \right) = -\frac{2}{N} \sum_{i=1}^{N} (\text{실제값}_i - \text{예측값}_i)$$

**업데이트 식**
- 편미분 값이 너무 클 수 있기 때문에 보정 계수 $\eta$를 곱하는데, 이를 **학습률**이라고 한다.
- 새로운 $w$는 이전 $w$에서 편미분 결괏값을 마이너스(-)하면서 적용한다.

$$w_1 \leftarrow w_1 + \eta \frac{2}{N} \sum_{i=1}^{N} x_i (\text{실제값}_i - \text{예측값}_i), \qquad w_0 \leftarrow w_0 + \eta \frac{2}{N} \sum_{i=1}^{N} (\text{실제값}_i - \text{예측값}_i)$$

**경사 하강법의 일반적인 프로세스**
1. $w_0, w_1$를 임의의 값으로 설정하고 첫 비용 함수의 값을 계산한다.
2. 위 업데이트 식으로 $w_1, w_0$를 업데이트한 후 다시 비용 함수의 값을 계산한다.
3. 비용 함수의 값이 감소했으면 다시 2단계를 반복한다. 더 이상 비용 함수의 값이 감소하지 않으면 그때의 $w_1, w_0$를 구하고 반복을 중지한다.

### 실습: $y = 4X + 6$을 근사하는 데이터에 경사 하강법 적용

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

np.random.seed(0)
# y = 4X + 6을 근사(w1=4, w0=6). 임의의 값은 노이즈를 위해 만듦.
X = 2 * np.random.rand(100, 1)
y = 6 + 4 * X + np.random.randn(100, 1)

# X, y 데이터 세트 산점도로 시각화
plt.scatter(X, y)

- 비용 함수 `get_cost()`: 실제 y 값과 예측된 y 값을 인자로 받아 $\frac{1}{N}\sum(\text{실제값}_i - \text{예측값}_i)^2$을 계산해 반환한다.

In [ ]:
def get_cost(y, y_pred):
    N = len(y)
    cost = np.sum(np.square(y - y_pred)) / N
    return cost

- `get_weight_updates()`: $w_1$과 $w_0$를 모두 0으로 초기화한 뒤 업데이트할 값을 반환한다.
- 예측 배열 `y_pred`는 `np.dot(X, w1.T) + w0`로 구한다. 100개의 데이터 X(1, 2, …, 100)가 있다면 예측값은 $w_0 + X(1)w_1 + X(2)w_1 + \cdots + X(100)w_1$이며, 이는 입력 배열 X와 $w_1$ 배열의 내적과 같다.
- `w0_factors`는 $w_0$ 업데이트를 내적 연산으로 구하기 위해 모든 값이 1인 (N, 1) 행렬이다.

In [ ]:
# w1과 w0를 업데이트할 w1_update, w0_update를 반환.
def get_weight_updates(w1, w0, X, y, learning_rate=0.01):
    N = len(y)
    # 먼저 w1_update, w0_update를 각각 w1, w0의 shape와 동일한 크기를 가진 0 값으로 초기화
    w1_update = np.zeros_like(w1)
    w0_update = np.zeros_like(w0)
    # 예측 배열 계산하고 예측과 실제 값의 차이 계산
    y_pred = np.dot(X, w1.T) + w0
    diff = y - y_pred

    # w0_update를 dot 행렬 연산으로 구하기 위해 모두 1값을 가진 행렬 생성
    w0_factors = np.ones((N, 1))

    # w1과 w0을 업데이트할 w1_update와 w0_update 계산
    w1_update = -(2 / N) * learning_rate * (np.dot(X.T, diff))
    w0_update = -(2 / N) * learning_rate * (np.dot(w0_factors.T, diff))

    return w1_update, w0_update

In [ ]:
# 입력 인자 iters로 주어진 횟수만큼 반복적으로 w1과 w0를 업데이트 적용함.
def gradient_descent_steps(X, y, iters=10000):
    # w0와 w1을 모두 0으로 초기화.
    w0 = np.zeros((1, 1))
    w1 = np.zeros((1, 1))

    # 인자로 주어진 iters만큼 반복적으로 get_weight_updates() 호출해 w1, w0 업데이트 수행.
    for ind in range(iters):
        w1_update, w0_update = get_weight_updates(w1, w0, X, y, learning_rate=0.01)
        w1 = w1 - w1_update
        w0 = w0 - w0_update

    return w1, w0

In [ ]:
def get_cost(y, y_pred):
    N = len(y)
    cost = np.sum(np.square(y - y_pred)) / N
    return cost

w1, w0 = gradient_descent_steps(X, y, iters=1000)
print("w1:{0:.3f} w0:{1:.3f}".format(w1[0, 0], w0[0, 0]))
y_pred = w1[0, 0] * X + w0
print('Gradient Descent Total Cost:{0:.4f}'.format(get_cost(y, y_pred)))

- 실제 선형식인 $y = 4X + 6$과 유사하게 $w_1$은 약 4.022, $w_0$는 약 6.162가 도출되었다.

In [ ]:
plt.scatter(X, y)
plt.plot(X, y_pred)

### 확률적 경사 하강법(Stochastic Gradient Descent)

- 일반적인 경사 하강법은 모든 학습 데이터에 대해 반복적으로 비용 함수 최소화를 위한 값을 업데이트하기 때문에 **수행 시간이 매우 오래 걸린다**.
- **확률적 경사 하강법**은 전체 입력 데이터로 $w$가 업데이트되는 값을 계산하는 것이 아니라 **일부 데이터만 이용**해 $w$가 업데이트되는 값을 계산하므로 빠른 속도를 보장한다.
- 대용량 데이터의 경우 대부분 확률적 경사 하강법이나 **미니 배치 확률적 경사 하강법**을 이용해 최적 비용 함수를 도출한다.
- 아래 코드는 전체 X, y 데이터에서 랜덤하게 `batch_size`만큼 데이터를 추출해 이를 기반으로 `w1_update`, `w0_update`를 계산하는 부분만 차이가 있다.

In [ ]:
def stochastic_gradient_descent_steps(X, y, batch_size=10, iters=1000):
    w0 = np.zeros((1, 1))
    w1 = np.zeros((1, 1))
    prev_cost = 100000
    iter_index = 0

    for ind in range(iters):
        np.random.seed(ind)
        # 전체 X, y 데이터에서 랜덤하게 batch_size만큼 데이터를 추출해 sample_X, sample_y로 저장
        stochastic_random_index = np.random.permutation(X.shape[0])
        sample_X = X[stochastic_random_index[0:batch_size]]
        sample_y = y[stochastic_random_index[0:batch_size]]
        # 랜덤하게 batch_size만큼 추출된 데이터 기반으로 w1_update, w0_update 계산 후 업데이트
        w1_update, w0_update = get_weight_updates(w1, w0, sample_X, sample_y, learning_rate=0.01)
        w1 = w1 - w1_update
        w0 = w0 - w0_update

    return w1, w0

In [ ]:
w1, w0 = stochastic_gradient_descent_steps(X, y, iters=1000)
print("w1:", round(w1[0, 0], 3), "w0:", round(w0[0, 0], 3))
y_pred = w1[0, 0] * X + w0
print('Stochastic Gradient Descent Total Cost:{0:.4f}'.format(get_cost(y, y_pred)))

- 확률적 경사 하강법으로 구한 $w_0, w_1$은 경사 하강법으로 구한 값과 큰 차이가 없으며, 예측 오류 비용도 0.9937로 경사 하강법의 0.9935와 비교해 아주 조금 높을 뿐 성능 차이가 크지 않다.
- 따라서 큰 데이터를 처리할 경우에는 경사 하강법은 매우 시간이 오래 걸리므로 일반적으로 확률적 경사 하강법을 이용한다.

**피처가 여러 개인 경우**
- 피처가 M개 $(X_1, X_2, \dots, X_{100})$ 있다면 회귀 계수도 M+1개(1개는 $w_0$)로 도출되며, 예측 회귀식은 $\hat{Y} = w_0 + w_1 X_1 + w_2 X_2 + \cdots + w_{100} X_{100}$이 된다.
- 데이터의 개수가 N이고 피처가 M개인 입력 행렬을 $X_{mat}$, 회귀 계수 $w_1, \dots, w_{100}$을 배열 $W$라고 하면 예측 행렬은 $\hat{Y} = X_{mat} \cdot W^T + w_0$로 구할 수 있다.
- $w_0$를 $W$ 배열 안에 포함시키기 위해 $X_{mat}$의 맨 처음 열에 모든 데이터 값이 1인 피처 Feat 0을 추가하면 $\hat{Y} = X_{mat} \cdot W^T$로 표현할 수 있다.

## 04. 사이킷런 LinearRegression을 이용한 보스턴 주택 가격 예측

### 개념 정리

**LinearRegression 클래스 - Ordinary Least Squares**
- `LinearRegression` 클래스는 예측값과 실제 값의 RSS를 최소화해 OLS(Ordinary Least Squares) 추정 방식으로 구현한 클래스이다.
- `fit()` 메서드로 X, y 배열을 입력받으면 회귀 계수(Coefficients)인 $W$를 `coef_` 속성에 저장한다.

| 구분 | 이름 | 설명 |
|---|---|---|
| 입력 파라미터 | `fit_intercept` | 불린 값, 디폴트 True. 절편 값을 계산할 것인지 말지를 지정. False로 지정하면 절편이 사용되지 않고 0으로 지정됨 |
| 입력 파라미터 | `copy_X` | 디폴트 True. 입력 데이터를 복사해 사용할지 여부 |
| 입력 파라미터 | `n_jobs` | 병렬로 처리할 작업 수 |
| 속성 | `coef_` | `fit()` 메서드를 수행했을 때 회귀 계수가 배열 형태로 저장. Shape는 (Target 값 개수, 피처 개수) |
| 속성 | `intercept_` | 절편 값 |

- 책에는 `normalize` 파라미터(회귀 수행 전 입력 데이터 세트를 정규화)도 나오지만 사이킷런 1.2에서 삭제되었다. 정규화가 필요하면 `StandardScaler` 등을 따로 적용한다.

**다중 공선성(multi-collinearity) 문제**
- OLS 기반의 회귀 계수 계산은 입력 피처의 **독립성**에 많은 영향을 받는다.
- 피처 간의 상관관계가 매우 높은 경우 분산이 매우 커져서 오류에 매우 민감해지는데, 이를 다중 공선성 문제라고 한다.
- 일반적으로 상관관계가 높은 피처가 많은 경우 독립적인 중요한 피처만 남기고 제거하거나 규제를 적용한다. 매우 많은 피처가 다중 공선성 문제를 가지고 있다면 PCA를 통해 차원 축소를 수행하는 것도 고려해 볼 수 있다.

#### [코드 수정 사항 반영] p.321 LinearRegression 클래스 선언

책의 `LinearRegression(fit_intercept=True, normalize=False, copy_X=True, n_jobs=1)`에서 삭제된 `normalize`를 빼고 아래와 같이 선언한다.

In [ ]:
from sklearn.linear_model import LinearRegression
linear_regression_model = LinearRegression(fit_intercept=True, n_jobs=1)

### 회귀 평가 지표

회귀의 평가를 위한 지표는 실제 값과 회귀 예측값의 차이 값을 기반으로 한다. 실제 값과 예측값의 차이를 그냥 더하면 +와 -가 섞여서 오류가 상쇄되기 때문에 오류의 절댓값 평균이나 제곱, 또는 제곱한 뒤 다시 루트를 씌운 평균값을 구한다.

| 평가 지표 | 설명 | 수식 |
|---|---|---|
| MAE | Mean Absolute Error. 실제 값과 예측값의 차이를 절댓값으로 변환해 평균한 것 | $\frac{1}{n}\sum_{i=1}^{n} \lvert Y_i - \hat{Y}_i \rvert$ |
| MSE | Mean Squared Error. 실제 값과 예측값의 차이를 제곱해 평균한 것 | $\frac{1}{n}\sum_{i=1}^{n} (Y_i - \hat{Y}_i)^2$ |
| RMSE | MSE 값은 오류의 제곱을 구하므로 실제 오류 평균보다 더 커지는 특성이 있어 MSE에 루트를 씌운 것 | $\sqrt{\frac{1}{n}\sum_{i=1}^{n} (Y_i - \hat{Y}_i)^2}$ |
| $R^2$ | 분산 기반으로 예측 성능을 평가. 실제 값의 분산 대비 예측값의 분산 비율을 지표로 하며, 1에 가까울수록 예측 정확도가 높음 | $\frac{\text{예측값 Variance}}{\text{실제값 Variance}}$ |

- 이 밖에 MSE나 RMSE에 로그를 적용한 MSLE(Mean Squared Log Error)와 RMSLE(Root Mean Squared Log Error)도 사용한다.

**사이킷런의 평가 지표 API와 Scoring 파라미터**

| 평가 방법 | 사이킷런 평가 지표 API | Scoring 함수 적용 값 |
|---|---|---|
| MAE | `metrics.mean_absolute_error` | `'neg_mean_absolute_error'` |
| MSE | `metrics.mean_squared_error` | `'neg_mean_squared_error'` |
| RMSE | `metrics.mean_squared_error`를 그대로 사용하되 `squared` 파라미터를 False로 설정 | `'neg_root_mean_squared_error'` |
| MSLE | `metrics.mean_squared_log_error` | `'neg_mean_squared_log_error'` |
| $R^2$ | `metrics.r2_score` | `'r2'` |

- 최신 사이킷런에서는 `squared` 파라미터가 삭제되었고 `metrics.root_mean_squared_error`로 RMSE를 바로 구할 수 있다. 아래 실습에서는 책처럼 MSE에 `np.sqrt()`를 적용한다.
- `cross_val_score`, `GridSearchCV` 같은 Scoring 함수에 회귀 평가 지표를 적용할 때 **'neg_'** 가 붙는 이유: 사이킷런의 Scoring 함수는 **score 값이 클수록 좋은 평가 결과**로 자동 평가한다. 그런데 실제 값과 예측값의 오류 차이를 기반으로 하는 회귀 평가 지표는 값이 커지면 오히려 나쁜 모델이라는 의미이므로, 원래 평가 지표 값에 -1을 곱해서 음수(Negative)를 만들어 작은 오류 값이 더 큰 숫자로 인식되게 한다.

### LinearRegression을 이용해 보스턴 주택 가격 회귀 구현

**보스턴 주택 가격 데이터 피처**

| 피처 | 의미 |
|---|---|
| CRIM | 지역별 범죄 발생률 |
| ZN | 25,000평방피트를 초과하는 거주 지역의 비율 |
| INDUS | 비상업 지역 넓이 비율 |
| CHAS | 찰스강에 대한 더미 변수(강의 경계에 위치한 경우는 1, 아니면 0) |
| NOX | 일산화질소 농도 |
| RM | 거주할 수 있는 방 개수 |
| AGE | 1940년 이전에 건축된 소유 주택의 비율 |
| DIS | 5개 주요 고용센터까지의 가중 거리 |
| RAD | 고속도로 접근 용이도 |
| TAX | 10,000달러당 재산세율 |
| PTRATIO | 지역의 교사와 학생 수 비율 |
| B | 지역의 흑인 거주 비율 |
| LSTAT | 하위 계층의 비율 |
| MEDV (→ PRICE) | 본인 소유의 주택 가격(중앙값) |

#### [코드 수정 사항 반영] p.324 Dataset 불러오기

`load_boston()`이 사이킷런 1.2에서 삭제되어 CMU 원본 데이터를 직접 읽는다. 원본 파일은 한 레코드가 두 줄에 나뉘어 있어서, 짝수 행 전체와 홀수 행 앞 2개 컬럼을 이어 붙여 13개 피처를 만들고 홀수 행 3번째 컬럼을 타깃(PRICE)으로 쓴다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats
%matplotlib inline

# Dataset 불러오기
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

feature_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
bostonDF = pd.DataFrame(data, columns=feature_names)
bostonDF['PRICE'] = target

print('Boston 데이터 세트 크기 :', bostonDF.shape)
display(bostonDF.head())

- 각 컬럼이 회귀 결과에 미치는 영향이 어느 정도인지 시각화해서 알아본다. 8개 피처에 대해 시본의 `regplot()`으로 산점도와 선형 회귀 직선을 함께 표현한다.
- 맷플롯립의 `subplots()`는 여러 개의 그래프를 한 번에 표현하기 위해 사용하며, `ncols`는 열 방향, `nrows`는 행 방향으로 위치할 그래프의 개수이다.

In [ ]:
# 2개의 행과 4개의 열을 가진 subplots를 이용. axs는 4x2개의 ax를 가짐.
fig, axs = plt.subplots(figsize=(16, 8), ncols=4, nrows=2)
lm_features = ['RM', 'ZN', 'INDUS', 'NOX', 'AGE', 'PTRATIO', 'LSTAT', 'RAD']
for i, feature in enumerate(lm_features):
    row = int(i / 4)
    col = i % 4
    # 시본의 regplot을 이용해 산점도와 선형 회귀 직선을 함께 표현
    sns.regplot(x=feature, y='PRICE', data=bostonDF, ax=axs[row][col])

- 다른 컬럼보다 **RM**과 **LSTAT**의 PRICE 영향도가 가장 두드러지게 나타난다.
- RM(방 개수)은 양 방향의 선형성(Positive Linearity)이 가장 크다. 방의 크기가 클수록 가격이 증가하는 모습을 확연히 보여준다.
- LSTAT(하위 계층의 비율)는 음 방향의 선형성(Negative Linearity)이 가장 크다. LSTAT이 적을수록 PRICE가 증가하는 모습이 확연히 나타난다.

- 이제 `LinearRegression` 클래스를 이용해 보스턴 주택 가격의 회귀 모델을 만든다. `train_test_split()`으로 학습과 테스트 데이터 세트를 분리해 학습과 예측을 수행하고, `mean_squared_error()`와 `r2_score()`로 MSE와 R2 Score를 측정한다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

y_target = bostonDF['PRICE']
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)

X_train, X_test, y_train, y_test = train_test_split(X_data, y_target, test_size=0.3, random_state=156)

# Linear Regression OLS로 학습/예측/평가 수행.
lr = LinearRegression()
lr.fit(X_train, y_train)
y_preds = lr.predict(X_test)
mse = mean_squared_error(y_test, y_preds)
rmse = np.sqrt(mse)

print('MSE : {0:.3f} , RMSE : {1:.3F}'.format(mse, rmse))
print('Variance score : {0:.3f}'.format(r2_score(y_test, y_preds)))

In [ ]:
print('절편 값:', lr.intercept_)
print('회귀 계수값:', np.round(lr.coef_, 1))

- `coef_` 속성은 회귀 계수 값만 가지고 있으므로, 이를 피처별 회귀 계수 값으로 다시 매핑하고 높은 값 순으로 출력한다. 판다스 Series의 `sort_values()`를 이용한다.

In [ ]:
# 회귀 계수를 큰 값 순으로 정렬하기 위해 Series로 생성. 인덱스 컬럼명에 유의
coeff = pd.Series(data=np.round(lr.coef_, 1), index=X_data.columns)
coeff.sort_values(ascending=False)

- RM이 양의 값으로 회귀 계수가 가장 크며, NOX 피처의 회귀 계수 - 값이 너무 커 보인다. 이후 최적화를 수행하면서 피처 coefficients의 변화도 같이 살펴볼 것이다.
- 이번에는 5개의 폴드 세트에서 `cross_val_score()`를 이용해 교차 검증으로 MSE와 RMSE를 측정한다.
- `cross_val_score(scoring='neg_mean_squared_error')`로 반환된 값은 모두 음수이므로, 여기에 -1을 곱해 양의 값인 원래 모델에서 계산된 MSE 값으로 바꾼 뒤 `np.sqrt()`로 RMSE를 구한다.

In [ ]:
from sklearn.model_selection import cross_val_score

y_target = bostonDF['PRICE']
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)
lr = LinearRegression()

# cross_val_score()로 5 폴드 세트로 MSE를 구한 뒤 이를 기반으로 다시 RMSE 구함.
neg_mse_scores = cross_val_score(lr, X_data, y_target, scoring="neg_mean_squared_error", cv=5)
rmse_scores = np.sqrt(-1 * neg_mse_scores)
avg_rmse = np.mean(rmse_scores)

# cross_val_score(scoring="neg_mean_squared_error")로 반환된 값은 모두 음수
print(' 5 folds 의 개별 Negative MSE scores: ', np.round(neg_mse_scores, 2))
print(' 5 folds 의 개별 RMSE scores : ', np.round(rmse_scores, 2))
print(' 5 folds 의 평균 RMSE : {0:.3f} '.format(avg_rmse))

- 5개 폴드 세트에 대해서 교차 검증을 수행한 결과, 평균 RMSE는 약 5.829가 나왔다.

## 05. 다항 회귀와 과(대)적합/과소적합 이해

### 개념 정리

**다항(Polynomial) 회귀**
- 지금까지 설명한 회귀는 $y = w_0 + w_1 x_1 + w_2 x_2 + \cdots + w_n x_n$과 같이 독립변수(feature)와 종속변수(target)의 관계가 일차 방정식 형태로 표현된 회귀였다.
- 회귀가 독립변수의 단항식이 아닌 2차, 3차 방정식과 같은 다항식으로 표현되는 것을 **다항 회귀**라고 한다.

$$y = w_0 + w_1 x_1 + w_2 x_2 + w_3 x_1 x_2 + w_4 x_1^2 + w_5 x_2^2$$

- 주의: **다항 회귀는 선형 회귀이다.** 회귀에서 선형 회귀/비선형 회귀를 나누는 기준은 **회귀 계수가 선형/비선형인지**에 따른 것이지 독립변수의 선형/비선형 여부와는 무관하다.
- 위 식에서 새로운 변수 $z = [x_1, x_2, x_1 x_2, x_1^2, x_2^2]$라고 하면 $y = w_0 + w_1 z_1 + w_2 z_2 + w_3 z_3 + w_4 z_4 + w_5 z_5$와 같이 표현할 수 있으므로 여전히 선형 회귀이다.
- 사이킷런은 다항 회귀를 위한 클래스를 명시적으로 제공하지 않는다. 대신 다항 회귀 역시 선형 회귀이기 때문에, **비선형 함수를 선형 모델에 적용시키는 방법**으로 구현한다.
- `PolynomialFeatures` 클래스로 피처를 다항식 피처로 변환한다. `degree` 파라미터로 입력받은 단항식 피처를 degree에 해당하는 다항식 피처로 변환하며, 다른 전처리 변환 클래스와 마찬가지로 `fit()`, `transform()` 메서드로 변환 작업을 수행한다.

- 단항 값 $[x_1, x_2]$를 2차 다항 값으로 변환하면 $[1, x_1, x_2, x_1^2, x_1 x_2, x_2^2]$가 된다.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

# 다항식으로 변환한 단항식 생성, [[0,1],[2,3]]의 2X2 행렬 생성
X = np.arange(4).reshape(2, 2)
print('일차 단항식 계수 피처:\n', X)

# degree = 2인 2차 다항식으로 변환하기 위해 PolynomialFeatures를 이용해 변환
poly = PolynomialFeatures(degree=2)
poly.fit(X)
poly_ftr = poly.transform(X)
print('변환된 2차 다항식 계수 피처:\n', poly_ftr)

- 첫 번째 입력 단항 계수 피처 $[x_1=0, x_2=1]$은 $[1, x_1=0, x_2=1, x_1^2=0, x_1x_2=0, x_2^2=1]$ 형태인 $[1, 0, 1, 0, 0, 1]$로 변환된다.
- 두 번째 입력 단항 계수 피처 $[x_1=2, x_2=3]$은 $[1, 2, 3, 4, 6, 9]$로 변환된다.

이렇게 변환된 Polynomial 피처에 선형 회귀를 적용해 다항 회귀를 구현한다. 3차 다항 회귀 함수 $y = 1 + 2x_1 + 3x_1^2 + 4x_2^3$을 임의로 설정하고 이 회귀식의 계수를 예측해 본다.

In [ ]:
def polynomial_func(X):
    y = 1 + 2 * X[:, 0] + 3 * X[:, 0] ** 2 + 4 * X[:, 1] ** 3
    return y

X = np.arange(4).reshape(2, 2)
print('일차 단항식 계수 feature: \n', X)
y = polynomial_func(X)
print('삼차 다항식 결정값: \n', y)

In [ ]:
# 3차 다항식 변환
poly_ftr = PolynomialFeatures(degree=3).fit_transform(X)
print('3차 다항식 계수 feature: \n', poly_ftr)

# Linear Regression에 3차 다항식 계수 feature와 3차 다항식 결정값으로 학습 후 회귀 계수 확인
model = LinearRegression()
model.fit(poly_ftr, y)
print('Polynomial 회귀 계수\n', np.round(model.coef_, 2))
print('Polynomial 회귀 Shape :', model.coef_.shape)

- 일차 단항식 계수 피처는 2개였지만, 3차 다항식 Polynomial 변환 이후에는 다항식 계수 피처가 10개로 늘어난다.
- 이 피처 데이터 세트에 `LinearRegression`으로 3차 다항 회귀 함수식을 회귀 분석한 결과, 회귀 계수는 [0, 0.18, 0.18, 0.36, 0.54, 0.72, 0.72, 1.08, 1.62, 2.34]로 10개가 되었다.
- 원래 다항식 $y = 1 + 2x_1 + 3x_1^2 + 4x_2^3$의 계수 값인 [1, 2, 0, 3, 0, 0, 0, 0, 0, 4]와는 차이가 있지만 다항 회귀로 근사하고 있음을 알 수 있다.
- 이처럼 사이킷런은 `PolynomialFeatures`로 피처를 변환한 후에 `LinearRegression` 클래스로 다항 회귀를 구현한다.

- 피처 변환과 선형 회귀 적용을 각각 별도로 하는 것보다는 사이킷런의 `Pipeline` 객체를 이용해 한 번에 다항 회귀를 구현하는 것이 코드를 더 명료하게 작성하는 방법이다.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
import numpy as np

def polynomial_func(X):
    y = 1 + 2 * X[:, 0] + 3 * X[:, 0] ** 2 + 4 * X[:, 1] ** 3
    return y

# Pipeline 객체로 Streamline하게 Polynomial Feature 변환과 Linear Regression을 연결
model = Pipeline([('poly', PolynomialFeatures(degree=3)),
                  ('linear', LinearRegression())])
X = np.arange(4).reshape(2, 2)
y = polynomial_func(X)

model = model.fit(X, y)
print('Polynomial 회귀 계수\n', np.round(model.named_steps['linear'].coef_, 2))

### 다항 회귀를 이용한 과소적합 및 과적합 이해

- 다항 회귀는 피처의 직선적 관계가 아닌 복잡한 다항 관계를 모델링할 수 있다. 다항식의 차수가 높아질수록 매우 복잡한 피처 간의 관계까지 모델링이 가능하다.
- 하지만 다항 회귀의 **차수(degree)를 높일수록 학습 데이터에만 너무 맞춘 학습이 이뤄져서** 정작 테스트 데이터 환경에서는 오히려 예측 정확도가 떨어진다. 즉, 차수가 높아질수록 과적합의 문제가 크게 발생한다.

아래는 사이킷런 홈페이지의 예제를 이용한 실습이다.
- 원래 데이터 세트는 피처 X와 target y가 잡음(Noise)이 포함된 다항식의 코사인(Cosine) 그래프 관계를 가지도록 만든다.
- 이에 기반해 다항 회귀의 차수를 변화시키면서 그에 따른 회귀 예측 곡선과 예측 정확도를 비교한다.
- 학습 데이터는 30개의 임의의 데이터인 X, 그리고 X의 코사인 값에서 약간의 잡음 변동 값을 더한 target인 y로 구성된다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
%matplotlib inline

# 임의의 값으로 구성된 X 값에 대해 코사인 변환 값을 반환.
def true_fun(X):
    return np.cos(1.5 * np.pi * X)

# X는 0부터 1까지 30개의 임의의 값을 순서대로 샘플링한 데이터
np.random.seed(0)
n_samples = 30
X = np.sort(np.random.rand(n_samples))

# y 값은 코사인 기반의 true_fun()에서 약간의 노이즈 변동 값을 더한 값
y = true_fun(X) + np.random.randn(n_samples) * 0.1

- 다항식 차수를 각각 1, 4, 15로 변경하면서 예측 결과를 비교한다.
- 다항식 차수별로 학습을 수행한 뒤 `cross_val_score()`로 MSE 값을 구해 차수별 예측 성능을 평가한다.
- 0부터 1까지 균일하게 구성된 100개의 테스트용 데이터 세트를 이용해 차수별 회귀 예측 곡선을 그려본다.

In [ ]:
plt.figure(figsize=(14, 5))
degrees = [1, 4, 15]

# 다항 회귀의 차수(degree)를 1, 4, 15로 각각 변화시키면서 비교합니다.
for i in range(len(degrees)):
    ax = plt.subplot(1, len(degrees), i + 1)
    plt.setp(ax, xticks=(), yticks=())

    # 개별 degree별로 Polynomial 변환합니다.
    polynomial_features = PolynomialFeatures(degree=degrees[i], include_bias=False)
    linear_regression = LinearRegression()
    pipeline = Pipeline([("polynomial_features", polynomial_features),
                         ("linear_regression", linear_regression)])
    pipeline.fit(X.reshape(-1, 1), y)

    # 교차 검증으로 다항 회귀를 평가합니다.
    scores = cross_val_score(pipeline, X.reshape(-1, 1), y, scoring="neg_mean_squared_error", cv=10)
    # Pipeline을 구성하는 세부 객체를 접근하는 named_steps['객체명']을 이용해 회귀계수 추출
    coefficients = pipeline.named_steps['linear_regression'].coef_
    print('\nDegree {0} 회귀 계수는 {1} 입니다.'.format(degrees[i], np.round(coefficients, 2)))
    print('Degree {0} MSE 는 {1} 입니다.'.format(degrees[i], -1 * np.mean(scores)))

    # 0부터 1까지 테스트 데이터 세트를 100개로 나눠 예측을 수행합니다.
    # 테스트 데이터 세트에 회귀 예측을 수행하고 예측 곡선과 실제 곡선을 그려서 비교합니다.
    X_test = np.linspace(0, 1, 100)
    # 예측값 곡선
    plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="Model")
    # 실제 값 곡선
    plt.plot(X_test, true_fun(X_test), '--', label="True function")
    plt.scatter(X, y, edgecolor='b', s=20, label="Samples")

    plt.xlabel("x"); plt.ylabel("y"); plt.xlim((0, 1)); plt.ylim((-2, 2)); plt.legend(loc="best")
    plt.title("Degree {}\nMSE = {:.2e}(+/- {:.2e})".format(degrees[i], -scores.mean(), scores.std()))

plt.show()

- 실선으로 표현된 예측 곡선은 다항 회귀 예측 곡선이고, 점선으로 표현된 곡선은 실제 데이터 세트 X, y의 코사인 곡선이다. 학습 데이터는 0부터 1까지의 30개의 임의의 X 값과 그에 따른 코사인 Y 값에 잡음을 변동 값으로 추가해 구성했으며 파란색 점으로 표현했다.

| 차수 | 결과 | 해석 |
|---|---|---|
| Degree 1 | MSE 약 0.41 | 단순한 직선으로서 단순 선형 회귀와 같다. 실제 데이터 세트인 코사인 데이터 세트를 직선으로 예측하기에는 너무 단순해 보인다. 예측 곡선이 학습 데이터의 패턴을 제대로 반영하지 못하고 있는 **과소적합** 모델이다 |
| Degree 4 | MSE 약 0.04 | 실제 데이터 세트와 유사한 모습이다. 변동하는 잡음까지 예측하지는 못했지만 학습 데이터 세트를 비교적 잘 반영해 코사인 곡선 기반으로 테스트 데이터를 잘 예측한 곡선을 가진 모델이 되었다. **가장 뛰어난 예측 성능** |
| Degree 15 | 책: MSE 약 182,815,433 (엄청나게 큰 값) / 최신 사이킷런 실행 시 약 0.22 (Degree 4의 5배 이상) | 예측 곡선이 데이터 세트의 변동 잡음 값까지 지나치게 반영한 결과, 예측 곡선이 학습 데이터 세트만 정확히 예측하고 테스트 값의 실제 곡선과는 완전히 다른 형태가 되었다. 회귀 계수도 매우 큰 값으로 커졌다. 학습 데이터에 너무 충실하게 맞춘 **과적합**이 심한 모델 |

- 좋은 예측 모델은 Degree 1과 같이 학습 데이터의 패턴을 지나치게 단순화한 과소적합 모델도 아니고, Degree 15와 같이 모든 학습 데이터의 패턴을 하나하나 감안한 지나치게 복잡한 과적합 모델도 아닌, **학습 데이터의 패턴을 잘 반영하면서도 복잡하지 않은 균형 잡힌(Balanced) 모델**을 의미한다.

### 편향-분산 트레이드오프(Bias-Variance Trade off)

- 편향-분산 트레이드오프는 머신러닝이 극복해야 할 가장 중요한 이슈 중 하나이다.
- 앞의 Degree 1과 같은 모델은 매우 단순화된 모델로서 지나치게 한 방향성으로 치우친 경향이 있으며, 이런 모델을 **고편향(High Bias)** 성을 가졌다고 표현한다.
- 반대로 Degree 15와 같은 모델은 학습 데이터 하나하나의 특성을 반영하면서 매우 복잡한 모델이 되었고 지나치게 높은 변동성을 가지게 되었다. 이런 모델을 **고분산(High Variance)** 성을 가졌다고 표현한다.

**양궁 과녁 그래프로 이해하기**

| | 저분산(Low Variance) | 고분산(High Variance) |
|---|---|---|
| **저편향(Low Bias)** | 예측 결과가 실제 결과에 매우 잘 근접하면서도 예측 변동이 크지 않고 특정 부분에 집중된 아주 뛰어난 성능 | 예측 결과가 실제 결과에 비교적 근접하지만, 예측 결과가 실제 결과를 중심으로 꽤 넓은 부분에 분포 |
| **고편향(High Bias)** | 정확한 결과에서 벗어나면서도 예측이 특정 부분에 집중 | 정확한 예측 결과를 벗어나면서도 넓은 부분에 분포 |

- 일반적으로 편향과 분산은 **한쪽이 높으면 한쪽이 낮아지는 경향**이 있다. 즉, 편향이 높으면 분산은 낮아지고(과소적합), 반대로 분산이 높으면 편향이 낮아진다(과적합).
- 편향이 너무 높으면 전체 오류가 높다. 편향을 점점 낮추면 동시에 분산이 높아지고 전체 오류도 낮아지게 된다.
- 편향을 낮추고 분산을 높이면서 전체 오류가 가장 낮아지는 **'골디락스' 지점**을 통과하면서 분산을 지속적으로 높이면 전체 오류 값이 오히려 증가하면서 예측 성능이 다시 저하된다.
- 높은 편향/낮은 분산에서 과소적합되기 쉬우며 낮은 편향/높은 분산에서 과적합되기 쉽다.
- 편향과 분산이 서로 트레이드오프를 이루면서 **오류 Cost 값이 최대로 낮아지는 모델을 구축하는 것**이 가장 효율적인 머신러닝 예측 모델을 만드는 방법이다.